In [1]:
import os
import random
import numpy as np
from PIL import Image

# 1. Simulate the FruitsInAnazon Dataset structure
train_dir = "FruitsInAnazon/train"
class_names = ['acai', 'cupuacu', 'graviola', 'guarana', 'pupunha', 'tucuma']

for fruit in class_names:
    os.makedirs(os.path.join(train_dir, fruit), exist_ok=True)
    # Create 15 dummy images per class as per Worksheet Figure 1
    for i in range(15):
        img = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
        img.save(os.path.join(train_dir, fruit, f"{fruit}_{i}.jpg"))

print(f"Found {len(os.listdir(train_dir))} classes: {sorted(os.listdir(train_dir))}")

Found 6 classes: ['acai', 'cupuacu', 'graviola', 'guarana', 'pupunha', 'tucuma']


In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Configuration
image_size = (224, 224)
batch_size = 32

# Load Dataset
train_ds, val_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="both",
    seed=1337,
    image_size=image_size,
    batch_size=batch_size,
)

# Data Augmentation Layers
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
])

Found 90 files belonging to 6 classes.
Using 72 files for training.
Using 18 files for validation.


In [3]:
model_scratch = keras.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    layers.Rescaling(1./255), # Recommended practice

    # Block 1
    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(class_names), activation='softmax')
])

model_scratch.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_scratch.summary() #

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 200704)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    25,690,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,710,790 (98.08 MB)

 Trainable params: 25,710,598 (98.08 MB)

 Non-trainable params: 192 (768.00 B)

In [4]:
from tensorflow.keras.applications import VGG16

# 1. Load Pre-trained Base
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. Freeze the base layers
for layer in base_model.layers:
    layer.trainable = False

# 3. Add Custom Top Layers
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(1024, activation='relu')(x)
output = layers.Dense(len(class_names), activation='softmax')(x)

# 4. Create and Compile
model_vgg = keras.Model(inputs=base_model.input, outputs=output)
model_vgg.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 5. Fit the model
# history = model_vgg.fit(train_ds, validation_data=val_ds, epochs=10)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
history = model_vgg.fit(train_ds, validation_data=val_ds, epochs=10)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 36s 11s/step - accuracy: 0.1111 - loss: 4.0709 - val_accuracy: 0.0556 - val_loss: 5.0080
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 37s 13s/step - accuracy: 0.1389 - loss: 5.6093 - val_accuracy: 0.2222 - val_loss: 3.9776
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 41s 13s/step - accuracy: 0.1528 - loss: 3.9814 - val_accuracy: 0.0556 - val_loss: 4.4038
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 41s 13s/step - accuracy: 0.2639 - loss: 3.3092 - val_accuracy: 0.0556 - val_loss: 2.3669
Epoch 5/10
1/3 ━━━━━━━━━━━━━━━━━━━━ 31s 16s/step - accuracy: 0.2188 - loss: 2.0768